In [1]:

import random


In [2]:

def check_winner(b):
    wins = [
        (0,1,2), (3,4,5), (6,7,8),
        (0,3,6), (1,4,7), (2,5,8),
        (0,4,8), (2,4,6)
    ]

    for a, c, d in wins:
        if b[a] == b[c] == b[d] != " ":
            return b[a]

    return "Draw" if " " not in b else None


def show_board(b):
    for i in range(0, 9, 3):
        print(" | ".join(b[i:i+3]))
        if i < 6:
            print("--+---+--")


In [3]:

Q = {}

alpha, gamma, epsilon = 0.3, 0.9, 0.2


def choose_action(board):
    state = tuple(board)

    actions = [i for i in range(9) if board[i] == " "]

    if random.random() < epsilon or state not in Q:
        return random.choice(actions)

    return max(actions, key=lambda a: Q[state].get(a, 0))


def update_q(state, action, reward, next_state):
    Q.setdefault(state, {})
    Q.setdefault(next_state, {})

    old = Q[state].get(action, 0)
    future = max(Q[next_state].values(), default=0)

    Q[state][action] = old + alpha * (
        reward + gamma * future - old
    )


In [4]:

for episode in range(20000):
    board = [" "] * 9
    history = []

    while True:
        state = tuple(board)
        action = choose_action(board)

        board[action] = "X"
        history.append((state, action))

        result = check_winner(board)

        if result:
            reward = 1 if result == "X" else 0

            for s, a in reversed(history):
                update_q(s, a, reward, tuple(board))
                reward *= gamma

            break

        # Opponent plays randomly
        available = [
            i for i in range(9) if board[i] == " "
        ]

        board[random.choice(available)] = "O"

        result = check_winner(board)

        if result:
            reward = -1 if result == "O" else 0

            for s, a in reversed(history):
                update_q(s, a, reward, tuple(board))
                reward *= gamma

            break


In [ ]:

epsilon = 0

board = [" "] * 9

print("Enter positions from 0 to 8:")

print("0 | 1 | 2\n--+---+--\n3 | 4 | 5\n--+---+--\n6 | 7 | 8")

while not check_winner(board):

    action = choose_action(board)
    board[action] = "X"

    print("\nAgent (X):")
    show_board(board)

    if check_winner(board):
        break

    available = [
        i for i in range(9) if board[i] == " "
    ]

    if not available:
        break

    move = int(input("Your move (0-8): "))

    if move not in available:
        print("Invalid move! Try again.")
        board[action] = " "
        continue

    board[move] = "O"

    print("\nYou (O):")
    show_board(board)

print("\nResult:", check_winner(board))


Enter positions from 0 to 8:
0 | 1 | 2
--+---+--
3 | 4 | 5
--+---+--
6 | 7 | 8

Agent (X):
  |   |  
--+---+--
  | X |  
--+---+--
  |   |  
Your move (0-8): 1

You (O):
  | O |  
--+---+--
  | X |  
--+---+--
  |   |  

Agent (X):
X | O |  
--+---+--
  | X |  
--+---+--
  |   |  
Your move (0-8): 3

You (O):
X | O |  
--+---+--
O | X |  
--+---+--
  |   |  

Agent (X):
X | O |  
--+---+--
O | X |  
--+---+--
  | X |  
